# Push-T × JEPA + **SIGReg officiel** (LeJEPA) — face à DINO-WM (SR 0.90)

World model **JEPA conjoint** (encodeur ViT + dynamique conditionnée par l'action), anti-collapse = **SIGReg du package `lejepa`** (aucun slot, aucune reconstruction, aucun décodeur). Planification **purement latente** (protocole DINO-WM).

Workflow : **1** (code+deps) → **2** (Drive) → **3** collecte (une fois) → **4** entraînement → **5** reprise → **6** éval MPC/oracle/aléa (coût latent) → **7** éval coût latent-sans-agent.

In [ ]:
# 1) Code + dépendances (gym-pusht + lejepa officiel)
import os
if not os.path.exists('/content/jepa-physics-bench'):
    !git clone https://github.com/frpatry/jepa-physics-bench.git /content/jepa-physics-bench
!cd /content/jepa-physics-bench && git pull
%cd /content/jepa-physics-bench
!pip install --quiet gym-pusht 'pymunk<7'
!pip install -q lejepa || pip install -q git+https://github.com/rbalestr-lab/lejepa
import torch, lejepa
t = lejepa.univariate.EppsPulley(t_max=3, n_points=17)
f = lejepa.multivariate.SlicingUnivariateTest(univariate_test=t, num_slices=1024)
z = torch.randn(128,128, requires_grad=True); f(z).backward()
print('OK — lejepa grad ok =', z.grad is not None)

In [ ]:
# 2) Drive (données + checkpoints persistants)
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('pas de Drive:', e)

In [ ]:
# 3) COLLECTE (une fois — sauter si pusht_data.npz est déjà sur Drive)
#!python -u pusht_data.py --n 10000 --T 6

In [ ]:
# 4) ENTRAÎNEMENT depuis zéro (JEPA conjoint + SIGReg rw=1.0)
!python -u pusht_jepa.py --steps 30000 --bs 32 --rw 1.0

In [ ]:
# 5) REPRISE du checkpoint (architecture auto-lue)
#!python -u pusht_jepa.py --resume 1 --steps 20000

In [ ]:
# 6) ÉVAL — PLANIF latente pleine (fidèle DINO-WM) : MPC vs ORACLE vs aléatoire
#    but par image, ≤100 pas, succès = coverage officiel (>0.95)
!python -u pusht_jepa.py --load 1 --tasks 10 --cost latent

In [ ]:
# 7) ÉVAL — variante coût 'sans agent' (tokens bleus du but exclus)
!python -u pusht_jepa.py --load 1 --tasks 10 --cost latent_noagent

---
## Pipeline **2 ÉTAPES** (encodeur gelé + dynamique) — pusht_jepa2.py

Le JEPA conjoint (au-dessus) échoue : SIGReg gaussianise mais rend l'encodeur anti-lisse (copy 0.60 > moyenne 0.42). Parade (vision LeCun/DINO-WM) : **geler** une tête de perception et imaginer dedans. **A** encodeur I-JEPA+SIGReg → gelé, **B** dynamique sur latents gelés, **C** planif.

In [ ]:
# A) ÉTAPE 1 — encodeur I-JEPA masqué + SIGReg (puis GELÉ) — sauve pusht_enc.pt
!python -u pusht_jepa2.py --stage enc --enc_steps 30000

In [ ]:
# B) ÉTAPE 2 — dynamique sur latents GELÉS — sauve pusht_dyn.pt
#    (affiche d'abord la SMOOTHNESS de l'encodeur : copy<<moyenne = bon signe)
!python -u pusht_jepa2.py --stage dyn --dyn_steps 20000

In [ ]:
# C) ÉVAL — planif CEM latente (encodeur gelé + dynamique) : MPC vs ORACLE vs aléa
!python -u pusht_jepa2.py --stage eval --tasks 10 --cost latent

In [ ]:
# C') diagnostic rapide de l'encodeur gelé seul (copy vs moyenne vs cos), sans planifier
#!python -u pusht_jepa2.py --stage dyn --dyn_steps 0

### FRAMESKIP — horizon utile (le bloc bouge vraiment)

Probe : sur 4 pas denses le bloc bouge 1-14px → coût plat → MPC=random. Parade standard (DINO-WM) : action **maintenue K pas** → chaque pas de planif couvre K pas d'env → le T se déplace franchement. **L'encodeur gelé est réutilisé tel quel** ; seule la dynamique se réentraîne.

In [ ]:
# FS-1) données frameskip=5 (actions maintenues 5 pas) -> pusht_data_fs5.npz
!python -u pusht_data.py --n 8000 --T 6 --frameskip 5

In [ ]:
# FS-2) dynamique sur données frameskip (encodeur pusht_enc.pt GELÉ réutilisé)
!python -u pusht_jepa2.py --stage dyn --dyn_steps 20000 \
    --data /content/drive/MyDrive/pusht_data_fs5.npz --dyn_ckpt /content/drive/MyDrive/pusht_dyn_fs5.pt

In [ ]:
# FS-3) PROBE alignement coût-vrai en frameskip (le bloc doit enfin varier : σ dist grande)
!python -u pusht_jepa2.py --stage probe --frameskip 5 --cost latent_noagent \
    --dyn_ckpt /content/drive/MyDrive/pusht_dyn_fs5.pt

In [ ]:
# FS-4) ÉVAL frameskip : MPC vs random (40 pas planif x5 = 200 pas env)
!python -u pusht_jepa2.py --stage eval --tasks 10 --frameskip 5 --max_steps 40 \
    --policies mpc,random --cost latent_noagent --dyn_ckpt /content/drive/MyDrive/pusht_dyn_fs5.pt
# puis comparer avec --cost latent (option A) si besoin